[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/12.%20The%20Berth%20Allocation%20with%20Tidal%20Windows%20Problem/P12-Tier-5_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open In Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 12. The Berth Allocation with Tidal Windows Problem
## Tier 5 — Integrated Digital Twin Simulation

### Goal
Build a simulation of berth allocation in which vessel arrivals, predicted tides, weather, berth status, and resource information are updated over time. At each hourly step, a greedy allocation procedure prioritizes pending vessels and searches for a feasible berth and start time. The simulation records operational data and supports what-if scenarios.

### Inputs and outputs
The input is a synthetic 30-day port scenario with four berths and 247 vessels. Vessel drafts, arrival times, processing times, costs, and other characteristics are generated using the fixed random seed in the code. The simulation produces a schedule, time-series performance metrics, and plots. Scenario experiments create separate simulated states to examine selected disruptions.

### Solution method
This notebook uses a rolling, hourly simulation with a priority-based greedy allocation heuristic. At each time step, arrived but unscheduled vessels are ordered by demurrage cost and arrival time. For each vessel, the method checks berths in list order and searches forward in half-hour increments for a tide-feasible, weather-adjusted start that does not overlap another assignment or maintenance interval. Among feasible berths, it chooses the assignment with the lowest modeled cost, including a preference discount when applicable.

This is a heuristic, not an exact global optimization algorithm: choices made for one vessel can affect later assignments, and the method does not prove that the resulting schedule is globally optimal. The rolling simulation and its greedy allocation sequence are the computational method demonstrated here.

### Model assumptions and scope
- Tidal levels, weather, equipment failures, and resource updates are synthetic stochastic data; they are not live sensor feeds, NOAA observations, or validated forecasts.
- Time is measured in hours, vessel lengths and drafts in meters, tidal levels and berth depths in meters, and demurrage cost in the model's cost units. Processing time and waiting time are in hours.
- The tide model is a single sinusoidal cycle with measurement and prediction noise. Berth depth is the low-tide depth plus the predicted tide elevation above the model's low-tide reference, capped at the specified high-tide depth.
- Weather delays affect candidate start times. Resource availability is recorded for monitoring, but tugboats, pilots, and stevedores are not binding allocation constraints. Vessel length, fuel use, and most equipment-level resource statuses are also not allocation constraints.
- A vessel is counted as served only after its scheduled processing end time has passed within the simulated horizon. Assignments that extend beyond the horizon are not counted as completed.
- This example does not implement live data synchronization, a true digital twin connected to physical port systems, a static-plan comparison, or a validated predictive-control system.

### Mathematical description of the allocation heuristic
Let $V_t$ be the vessels that have arrived by time $t$ and do not yet have an assignment, and let $B$ be the set of berths. An assignment for vessel $i$ consists of a berth $b$ and a start time $s$. Its end time is $e=s+p_i$, where $p_i$ is the vessel's processing time. The search considers candidate times on a half-hour grid, starting no earlier than the current time or the vessel's arrival.

For each feasible candidate, the modeled assignment cost is

$$c_{ib}(s)=d_i\max(0,s-a_i)\,q_{ib},\qquad
q_{ib}=\begin{cases}0.9,& b=r_i,\\1,& b\ne r_i,\end{cases}$$

where $d_i$ is the vessel's demurrage-cost parameter, $a_i$ is its arrival time, and $r_i$ is its preferred berth when a preference is specified. A candidate is feasible only if the berth is operational, its maintenance interval does not overlap $[s,e)$, no existing assignment on that berth overlaps $[s,e)$, and predicted depth is sufficient at both the candidate and weather-adjusted start times. The heuristic processes vessels in descending demurrage cost, breaking ties by arrival time, and selects the feasible berth assignment with the lowest cost for each vessel.

In the code, `optimize_berth_allocation` implements the priority ordering and berth choice, `find_feasible_start_time` performs the time search, `is_berth_available` checks berth and schedule conflicts, and `calculate_weather_delay` supplies the weather adjustment. Since each vessel is considered in turn, this procedure is a greedy sequential heuristic rather than a joint minimization of total schedule cost.

### What to examine
Use the performance plots to inspect completed vessels, waiting times, berth utilization, and the difference between measured and predicted tidal levels. Scenario results show how the same allocation procedure behaves under the specific modeled changes. They are simulation outcomes, not evidence that a real port system would achieve the same performance.

In [ ]:
# Standard-library and scientific-computing tools used by the simulation.
import copy
import random
import time
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes the synthetic example reproducible when run from the top.
np.random.seed(42)
random.seed(42)


In [ ]:
@dataclass
class Vessel:
    """Vessel attributes used by the berth-allocation simulation."""

    id: int
    name: str
    draft: float
    length: float
    processing_time: float
    arrival_time: float
    demurrage_cost: float
    captain_preference: Optional[int] = None
    speed: float = 15.0
    fuel_consumption: float = 50.0


@dataclass
class Berth:
    """Berth properties, operational status, and recent utilization."""

    id: int
    length: float
    depth_low_tide: float
    depth_high_tide: float
    equipment_status: Dict[str, bool]
    maintenance_schedule: List[Tuple[float, float]]
    utilization_history: deque


@dataclass
class TidalData:
    """Synthetic measured and predicted tidal information."""

    timestamp: float
    water_level: float
    predicted_level: float
    confidence: float
    sensor_id: int


@dataclass
class WeatherData:
    """Synthetic weather conditions and modeled storm probability."""

    timestamp: float
    wind_speed: float
    wind_direction: float
    wave_height: float
    visibility: float
    storm_probability: float


@dataclass
class ResourceStatus:
    """Snapshot of modeled resource availability."""

    timestamp: float
    tugboats_available: int
    pilots_available: int
    stevedores_available: int
    equipment_status: Dict[str, bool]


@dataclass
class DigitalTwinState:
    """Complete state passed between the simulation components."""

    current_time: float
    vessels: List[Vessel]
    berths: List[Berth]
    tidal_history: List[TidalData]
    weather_history: List[WeatherData]
    resource_history: List[ResourceStatus]
    current_schedule: Dict[int, Tuple[int, float, float]]
    performance_metrics: Dict[str, float]


In [ ]:
def initialize_digital_twin() -> DigitalTwinState:
    """Create the synthetic port, vessel arrivals, and initial metrics."""
    berths = [
        Berth(
            id=0,
            length=400.0,
            depth_low_tide=12.0,
            depth_high_tide=16.5,
            equipment_status={"crane": True, "mooring": True, "safety": True},
            maintenance_schedule=[],
            utilization_history=deque(maxlen=168),
        ),
        Berth(
            id=1,
            length=350.0,
            depth_low_tide=11.0,
            depth_high_tide=15.5,
            equipment_status={"crane": True, "mooring": True, "safety": True},
            maintenance_schedule=[],
            utilization_history=deque(maxlen=168),
        ),
        Berth(
            id=2,
            length=450.0,
            depth_low_tide=13.5,
            depth_high_tide=18.0,
            equipment_status={"crane": True, "mooring": True, "safety": True},
            maintenance_schedule=[],
            utilization_history=deque(maxlen=168),
        ),
        Berth(
            id=3,
            length=380.0,
            depth_low_tide=10.5,
            depth_high_tide=15.0,
            equipment_status={"crane": True, "mooring": True, "safety": True},
            maintenance_schedule=[],
            utilization_history=deque(maxlen=168),
        ),
    ]

    vessel_names = [
        "MSC_Mediterranean",
        "COSCO_Shanghai",
        "Maersk_Edinburgh",
        "ONE_Efficiency",
        "Hapag_Lloyd",
        "Evergreen_Marine",
        "CMA_CGM",
        "PIL_Singapore",
    ]
    vessels = []
    for vessel_id in range(247):
        arrival_time = np.random.uniform(0, 30) * 24 + np.random.uniform(0, 24)
        vessels.append(
            Vessel(
                id=vessel_id,
                name=f"{vessel_names[vessel_id % len(vessel_names)]}_{vessel_id}",
                draft=np.random.uniform(10.5, 15.5),
                length=np.random.uniform(200, 400),
                processing_time=np.random.uniform(3, 8),
                arrival_time=arrival_time,
                demurrage_cost=np.random.uniform(5000, 15000),
                captain_preference=np.random.choice(
                    [None, 0, 1, 2, 3], p=[0.6, 0.1, 0.1, 0.1, 0.1]
                ),
                speed=np.random.uniform(12, 18),
                fuel_consumption=np.random.uniform(40, 60),
            )
        )

    vessels.sort(key=lambda vessel: vessel.arrival_time)
    performance_metrics = {
        "total_vessels_served": 0,
        "average_waiting_time": 0.0,
        "berth_utilization": 0.0,
        "tidal_violations": 0,
        "fuel_efficiency": 0.0,
        "customer_satisfaction": 0.0,
    }

    return DigitalTwinState(
        current_time=0.0,
        vessels=vessels,
        berths=berths,
        tidal_history=[],
        weather_history=[],
        resource_history=[],
        current_schedule={},
        performance_metrics=performance_metrics,
    )


digital_twin = initialize_digital_twin()
print("Synthetic digital twin initialized.")
print(f"Berths: {len(digital_twin.berths)}")
print(f"Vessels: {len(digital_twin.vessels)}")
print("Simulation horizon: 30 days (720 hours)")
print(f"First arrival: {digital_twin.vessels[0].arrival_time:.1f} hours")
print(f"Last arrival: {digital_twin.vessels[-1].arrival_time:.1f} hours")

In [ ]:
class TidalPredictionSubsystem:
    """Generate synthetic tidal measurements and depth predictions."""

    def __init__(self, initial_time: float):
        self.current_time = initial_time
        self.tidal_cycle = 12.42
        self.amplitude = 3.7
        self.mean_level = 2.5
        self.prediction_error = 0.15

    def generate_tidal_data(self, timestamp: float) -> TidalData:
        """Generate a noisy measurement and prediction at a timestamp."""
        phase = 2 * np.pi * timestamp / self.tidal_cycle
        actual_level = self.mean_level + self.amplitude * np.sin(phase)
        measured_level = actual_level + np.random.normal(0, 0.05)
        prediction_error = np.random.normal(0, self.prediction_error)
        predicted_level = actual_level + prediction_error
        confidence = max(
            0.5,
            1.0 - abs(prediction_error) / (2 * self.prediction_error),
        )
        return TidalData(
            timestamp=timestamp,
            water_level=measured_level,
            predicted_level=predicted_level,
            confidence=confidence,
            sensor_id=int(timestamp) % 5,
        )

    def get_tidal_depth(self, timestamp: float, berth: Berth) -> float:
        """Estimate berth depth from predicted level, with a high-tide cap."""
        predicted_level = self.generate_tidal_data(timestamp).predicted_level
        low_tide_reference = self.mean_level - self.amplitude
        rise_above_low_tide = max(0.0, predicted_level - low_tide_reference)
        available_depth = berth.depth_low_tide + rise_above_low_tide
        return min(available_depth, berth.depth_high_tide)

    def is_high_tide_window(
        self, timestamp: float, vessel_draft: float, berth: Berth
    ) -> bool:
        """Return whether predicted water depth can accommodate the draft."""
        return self.get_tidal_depth(timestamp, berth) >= vessel_draft


class VesselTrackingSubsystem:
    """Maintain synthetic position updates and arrival-time estimates."""

    def __init__(self, vessels: List[Vessel]):
        self.vessels = vessels
        self.vessels_by_id = {vessel.id: vessel for vessel in vessels}
        self.position_updates = {}
        self.eta_predictions = {}

    def update_vessel_position(self, vessel_id: int, timestamp: float) -> None:
        """Simulate an AIS-style position update for one vessel."""
        vessel = self.vessels_by_id[vessel_id]
        if timestamp < vessel.arrival_time - 6:
            distance = 100.0
            speed = vessel.speed * np.random.uniform(0.9, 1.1)
        elif timestamp < vessel.arrival_time:
            distance = (vessel.arrival_time - timestamp) * vessel.speed
            speed = vessel.speed * np.random.uniform(0.8, 1.0)
        else:
            distance = 0.0
            speed = 0.0

        self.position_updates[vessel_id] = (timestamp, distance, speed)
        if distance > 0 and speed > 0:
            eta = timestamp + distance / speed
            self.eta_predictions[vessel_id] = eta + np.random.normal(0, 0.5)
        else:
            self.eta_predictions[vessel_id] = vessel.arrival_time

    def get_predicted_arrival(self, vessel_id: int) -> float:
        """Return the latest ETA estimate, or the scheduled arrival time."""
        vessel = self.vessels_by_id[vessel_id]
        return self.eta_predictions.get(vessel_id, vessel.arrival_time)


class ResourceManagementSubsystem:
    """Track resource availability and scheduled resource maintenance."""

    def __init__(
        self, num_tugboats: int = 6, num_pilots: int = 8, num_stevedores: int = 50
    ):
        self.base_tugboats = num_tugboats
        self.base_pilots = num_pilots
        self.base_stevedores = num_stevedores
        self.maintenance_schedule = {}

    def update_resource_status(self, timestamp: float) -> ResourceStatus:
        """Return current resource counts and synthetic equipment status."""
        tugboats_available = self.base_tugboats
        pilots_available = self.base_pilots
        stevedores_available = self.base_stevedores

        for resource_type, schedule in self.maintenance_schedule.items():
            for start, end in schedule:
                if start <= timestamp < end:
                    if resource_type == "tugboat":
                        tugboats_available = max(0, tugboats_available - 1)
                    elif resource_type == "pilot":
                        pilots_available = max(0, pilots_available - 1)

        equipment_status = {
            "tugboat_1": np.random.random() > 0.02,
            "tugboat_2": np.random.random() > 0.02,
            "pilot_boat_1": np.random.random() > 0.01,
            "crane_1": np.random.random() > 0.005,
            "crane_2": np.random.random() > 0.005,
        }
        return ResourceStatus(
            timestamp=timestamp,
            tugboats_available=tugboats_available,
            pilots_available=pilots_available,
            stevedores_available=stevedores_available,
            equipment_status=equipment_status,
        )

    def schedule_maintenance(
        self, resource_type: str, start_time: float, duration: float
    ) -> None:
        """Add a half-open maintenance interval for a resource."""
        self.maintenance_schedule.setdefault(resource_type, []).append(
            (start_time, start_time + duration)
        )


print("Tide, vessel-tracking, and resource subsystems are defined.")

In [ ]:
class WeatherForecastingSubsystem:
    """Generate synthetic weather and estimate weather-related delay."""

    def __init__(self):
        self.storm_probability_base = 0.05
        self.seasonal_factor = 1.0

    def generate_weather_data(self, timestamp: float) -> WeatherData:
        """Generate weather conditions using simple periodic patterns."""
        hour_of_day = timestamp % 24
        day_of_year = (timestamp / 24) % 365
        base_wind = 10 + 5 * np.sin(2 * np.pi * hour_of_day / 24)
        seasonal_wind = 3 * np.sin(2 * np.pi * day_of_year / 365)
        wind_speed = max(
            0.0, base_wind + seasonal_wind + np.random.normal(0, 3)
        )
        wind_direction = np.random.uniform(0, 360)
        wave_height = max(
            0.0, 0.5 + wind_speed / 20 + np.random.normal(0, 0.3)
        )
        visibility = max(0.1, np.random.lognormal(1.0, 0.5))
        storm_factor = min(1.0, wind_speed / 30)
        storm_probability = min(
            1.0, self.storm_probability_base * storm_factor * self.seasonal_factor
        )
        return WeatherData(
            timestamp=timestamp,
            wind_speed=wind_speed,
            wind_direction=wind_direction,
            wave_height=wave_height,
            visibility=visibility,
            storm_probability=storm_probability,
        )

    def calculate_weather_delay(
        self, weather: WeatherData, vessel: Vessel
    ) -> float:
        """Estimate a nonnegative delay from the modeled weather conditions."""
        delay = 0.0
        if weather.wind_speed > 25:
            delay += (weather.wind_speed - 25) * 0.2
        if weather.wave_height > 2.5:
            delay += (weather.wave_height - 2.5) * 0.5
        if weather.visibility < 1.0:
            delay += (1.0 - weather.visibility) * 2.0
        if weather.storm_probability > 0.7:
            delay += np.random.exponential(6)
        return delay


class OptimizationEngine:
    """Apply the priority-based greedy berth-allocation heuristic."""

    def __init__(
        self,
        tidal_system: TidalPredictionSubsystem,
        vessel_system: VesselTrackingSubsystem,
        weather_system: WeatherForecastingSubsystem,
        resource_system: ResourceManagementSubsystem,
    ):
        self.tidal_system = tidal_system
        self.vessel_system = vessel_system
        self.weather_system = weather_system
        self.resource_system = resource_system

    def optimize_berth_allocation(
        self,
        current_time: float,
        pending_vessels: List[Vessel],
        berths: List[Berth],
        existing_schedule: Optional[Dict[int, Tuple[int, float, float]]] = None,
    ) -> Dict[int, Tuple[int, float, float]]:
        """Assign pending vessels sequentially using cost and feasibility checks."""
        schedule = {}
        reservations = dict(existing_schedule or {})
        sorted_vessels = sorted(
            pending_vessels,
            key=lambda vessel: (vessel.demurrage_cost, vessel.arrival_time),
            reverse=True,
        )

        for vessel in sorted_vessels:
            best_assignment = None
            best_cost = float("inf")
            for berth in berths:
                feasible_start = self.find_feasible_start_time(
                    vessel, berth, current_time, reservations
                )
                if feasible_start is None:
                    continue

                delay = max(0.0, feasible_start - vessel.arrival_time)
                cost = vessel.demurrage_cost * delay
                if vessel.captain_preference == berth.id:
                    cost *= 0.9

                if cost < best_cost:
                    end_time = feasible_start + vessel.processing_time
                    best_cost = cost
                    best_assignment = (berth.id, feasible_start, end_time)

            if best_assignment is not None:
                schedule[vessel.id] = best_assignment
                reservations[vessel.id] = best_assignment

        return schedule

    def is_berth_available(
        self,
        berth: Berth,
        start_time: float,
        end_time: float,
        schedule: Dict[int, Tuple[int, float, float]],
    ) -> bool:
        """Check berth status, maintenance, and reservation overlap."""
        if not all(berth.equipment_status.values()):
            return False

        for maintenance_start, maintenance_end in berth.maintenance_schedule:
            if start_time < maintenance_end and maintenance_start < end_time:
                return False

        for reserved_berth, reserved_start, reserved_end in schedule.values():
            if reserved_berth == berth.id:
                if start_time < reserved_end and reserved_start < end_time:
                    return False
        return True

    def find_feasible_start_time(
        self,
        vessel: Vessel,
        berth: Berth,
        current_time: float,
        schedule: Optional[Dict[int, Tuple[int, float, float]]] = None,
    ) -> Optional[float]:
        """Search forward in half-hour increments for a feasible start."""
        reservations = schedule or {}
        search_end = min(current_time + 48, 720)
        earliest_start = max(current_time, vessel.arrival_time)

        for candidate_time in np.arange(earliest_start, search_end, 0.5):
            if not self.tidal_system.is_high_tide_window(
                candidate_time, vessel.draft, berth
            ):
                continue

            weather = self.weather_system.generate_weather_data(candidate_time)
            weather_delay = self.weather_system.calculate_weather_delay(
                weather, vessel
            )
            adjusted_start = candidate_time + weather_delay
            adjusted_end = adjusted_start + vessel.processing_time

            if not self.tidal_system.is_high_tide_window(
                adjusted_start, vessel.draft, berth
            ):
                continue
            if self.is_berth_available(
                berth, adjusted_start, adjusted_end, reservations
            ):
                return adjusted_start

        return None


print("Weather and allocation subsystems are defined.")

In [ ]:
class DigitalTwinSimulator:
    """Coordinate hourly updates, allocation, and performance tracking."""

    def __init__(self, initial_state: DigitalTwinState):
        self.state = copy.deepcopy(initial_state)
        self.tidal_system = TidalPredictionSubsystem(self.state.current_time)
        self.vessel_system = VesselTrackingSubsystem(self.state.vessels)
        self.weather_system = WeatherForecastingSubsystem()
        self.resource_system = ResourceManagementSubsystem()
        self.optimization_engine = OptimizationEngine(
            self.tidal_system,
            self.vessel_system,
            self.weather_system,
            self.resource_system,
        )
        self.simulation_duration = 720.0
        self.time_step = 1.0
        self.performance_history = []

    def run_simulation(self, verbose: bool = True) -> DigitalTwinState:
        """Run the hourly rolling simulation for the full horizon."""
        if verbose:
            print("Starting digital twin simulation.")
            print(f"Duration: {self.simulation_duration:.0f} hours")
            print(f"Time step: {self.time_step:.1f} hour")

        self.schedule_maintenance_events()
        number_of_steps = int(self.simulation_duration / self.time_step)
        for step in range(number_of_steps):
            current_time = step * self.time_step
            self.state.current_time = current_time
            self.update_subsystems(current_time)
            self.run_optimization_cycle(current_time)
            self.update_performance_metrics(current_time)
            if verbose and step % 24 == 0:
                self.print_daily_report(current_time / 24)

        if verbose:
            print("Simulation completed.")
            self.print_final_report()
        return self.state

    def schedule_maintenance_events(self) -> None:
        """Add the modeled resource and berth maintenance intervals."""
        self.resource_system.schedule_maintenance("tugboat", 120, 8)
        self.resource_system.schedule_maintenance("tugboat", 360, 6)
        self.state.berths[1].maintenance_schedule.append((240, 248))
        self.state.berths[3].maintenance_schedule.append((480, 486))

    def update_subsystems(self, current_time: float) -> None:
        """Append current synthetic tide, weather, and resource observations."""
        self.state.tidal_history.append(
            self.tidal_system.generate_tidal_data(current_time)
        )
        self.state.weather_history.append(
            self.weather_system.generate_weather_data(current_time)
        )
        self.state.resource_history.append(
            self.resource_system.update_resource_status(current_time)
        )

        for vessel in self.state.vessels:
            if 0 <= vessel.arrival_time - current_time <= 24:
                self.vessel_system.update_vessel_position(vessel.id, current_time)

    def run_optimization_cycle(self, current_time: float) -> None:
        """Allocate arrived vessels that do not already have assignments."""
        pending_vessels = [
            vessel
            for vessel in self.state.vessels
            if vessel.arrival_time <= current_time
            and vessel.id not in self.state.current_schedule
        ]
        if pending_vessels:
            new_assignments = self.optimization_engine.optimize_berth_allocation(
                current_time,
                pending_vessels,
                self.state.berths,
                self.state.current_schedule,
            )
            self.state.current_schedule.update(new_assignments)

    def update_performance_metrics(self, current_time: float) -> None:
        """Record completed vessels, waiting time, and hourly berth occupancy."""
        completed = []
        for vessel in self.state.vessels:
            assignment = self.state.current_schedule.get(vessel.id)
            if assignment is not None and assignment[2] <= current_time + self.time_step:
                completed.append((vessel, assignment))

        metrics = copy.deepcopy(self.state.performance_metrics)
        metrics["total_vessels_served"] = len(completed)
        if completed:
            waiting_times = [
                max(0.0, assignment[1] - vessel.arrival_time)
                for vessel, assignment in completed
            ]
            metrics["average_waiting_time"] = float(np.mean(waiting_times))
        else:
            metrics["average_waiting_time"] = 0.0

        interval_start = current_time
        interval_end = current_time + self.time_step
        total_berth_hours = 0.0
        for berth in self.state.berths:
            occupied_hours = 0.0
            for berth_id, start, end in self.state.current_schedule.values():
                if berth_id == berth.id:
                    occupied_hours += max(
                        0.0,
                        min(end, interval_end) - max(start, interval_start),
                    )
            berth.utilization_history.append(occupied_hours / self.time_step)
            total_berth_hours += occupied_hours

        available_berth_hours = len(self.state.berths) * self.time_step
        metrics["berth_utilization"] = (
            total_berth_hours / available_berth_hours
            if available_berth_hours
            else 0.0
        )
        self.state.performance_metrics = metrics
        self.performance_history.append(
            {"timestamp": current_time, "metrics": copy.deepcopy(metrics)}
        )

    def print_daily_report(self, day: float) -> None:
        """Print a concise snapshot of completed service and occupancy."""
        metrics = self.state.performance_metrics
        print(
            f"Day {day:.0f}: Completed={metrics['total_vessels_served']:3d}, "
            f"Mean wait={metrics['average_waiting_time']:.1f} h, "
            f"Hourly berth utilization={metrics['berth_utilization'] * 100:.1f}%"
        )

    def print_final_report(self) -> None:
        """Print end-of-horizon performance and data-collection summaries."""
        metrics = self.state.performance_metrics
        print("\n" + "=" * 60)
        print("DIGITAL TWIN SIMULATION SUMMARY")
        print("=" * 60)
        print(f"Completed vessels: {metrics['total_vessels_served']}")
        print(
            f"Average waiting time of completed vessels: "
            f"{metrics['average_waiting_time']:.2f} hours"
        )
        print(
            f"Completed vessels per simulated day: "
            f"{metrics['total_vessels_served'] / 30:.1f}"
        )
        print("Mean berth utilization over the most recent history window:")
        for berth in self.state.berths:
            if berth.utilization_history:
                average_utilization = np.mean(berth.utilization_history) * 100
                print(f"  Berth {berth.id}: {average_utilization:.1f}%")
        print(f"Tidal observations: {len(self.state.tidal_history)}")
        print(f"Weather observations: {len(self.state.weather_history)}")
        print(f"Resource observations: {len(self.state.resource_history)}")


print("Hourly digital twin simulator is defined.")

In [ ]:
# Run the main synthetic 30-day scenario.
simulator = DigitalTwinSimulator(digital_twin)
print("Initializing the synthetic Port of Hamburg scenario.")
print(f"Simulation period: {simulator.simulation_duration / 24:.0f} days")
print(f"Vessels in the input scenario: {len(digital_twin.vessels)}")
print(f"Berths in the input scenario: {len(digital_twin.berths)}")

start_time = time.time()
final_state = simulator.run_simulation(verbose=True)
elapsed_time = time.time() - start_time
print(f"Wall-clock runtime: {elapsed_time:.2f} seconds")

In [ ]:
def analyze_digital_twin_performance(
    state: DigitalTwinState, performance_history: List[Dict]
) -> None:
    """Plot time series and summarize the completed simulation run."""
    if not performance_history:
        print("No performance observations are available.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    timestamps = [record["timestamp"] / 24 for record in performance_history]
    vessels_completed = [
        record["metrics"]["total_vessels_served"] for record in performance_history
    ]
    waiting_times = [
        record["metrics"]["average_waiting_time"] for record in performance_history
    ]
    berth_utilization = [
        record["metrics"]["berth_utilization"] * 100
        for record in performance_history
    ]

    ax1.set_title("Cumulative Completed Vessels")
    ax1.set_xlabel("Simulation time (days)")
    ax1.set_ylabel("Completed vessels")
    ax1.plot(timestamps, vessels_completed, color="blue", label="Completed")
    ax1.fill_between(timestamps, vessels_completed, alpha=0.2, color="blue")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.set_title("Mean Waiting Time of Completed Vessels")
    ax2.set_xlabel("Simulation time (days)")
    ax2.set_ylabel("Waiting time (hours)")
    ax2.plot(timestamps, waiting_times, color="red", label="Mean waiting time")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    ax3.set_title("Hourly Berth Utilization")
    ax3.set_xlabel("Simulation time (days)")
    ax3.set_ylabel("Utilization (%)")
    ax3.plot(timestamps, berth_utilization, color="green", label="All berths")
    ax3.axhline(80, color="black", linestyle="--", label="80% reference")
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    ax4.set_title("Measured and Predicted Tidal Levels")
    ax4.set_xlabel("Simulation time (hours)")
    ax4.set_ylabel("Water level (m)")
    ax4.grid(True, alpha=0.3)
    recent_tidal = state.tidal_history[-72:]
    if recent_tidal:
        tidal_times = [observation.timestamp for observation in recent_tidal]
        measured_levels = [observation.water_level for observation in recent_tidal]
        predicted_levels = [
            observation.predicted_level for observation in recent_tidal
        ]
        ax4.plot(tidal_times, measured_levels, label="Measured", color="blue")
        ax4.plot(
            tidal_times,
            predicted_levels,
            linestyle="--",
            label="Predicted",
            color="orange",
        )
        ax4.legend()

    fig.tight_layout()
    plt.show()

    final_metrics = state.performance_metrics
    total_completed = int(final_metrics["total_vessels_served"])
    waiting_values = np.asarray(waiting_times, dtype=float)
    utilization_values = np.asarray(berth_utilization, dtype=float)
    arrival_completion_rate = (
        total_completed / len(state.vessels) * 100 if state.vessels else 0.0
    )

    print("\nDIGITAL TWIN PERFORMANCE ANALYSIS")
    print(f"Completed vessels by the horizon: {total_completed}")
    print(f"Completed share of modeled arrivals: {arrival_completion_rate:.1f}%")
    print(
        "Average waiting time at the final simulation step: "
        f"{final_metrics['average_waiting_time']:.2f} hours"
    )
    print(
        "Average of the hourly mean-wait series: "
        f"{np.mean(waiting_values):.2f} hours"
    )
    if waiting_values.size:
        print(f"Maximum hourly mean wait: {np.max(waiting_values):.2f} hours")
        print(f"Minimum hourly mean wait: {np.min(waiting_values):.2f} hours")
        print(f"Standard deviation of hourly mean wait: {np.std(waiting_values):.2f} hours")
    if utilization_values.size:
        print(f"Mean hourly berth utilization: {np.mean(utilization_values):.1f}%")
        print(f"Maximum hourly berth utilization: {np.max(utilization_values):.1f}%")
        print(f"Minimum hourly berth utilization: {np.min(utilization_values):.1f}%")
    print(f"Tidal observations: {len(state.tidal_history)}")
    print(f"Weather observations: {len(state.weather_history)}")
    print(f"Resource observations: {len(state.resource_history)}")
    print(f"Hourly performance records: {len(performance_history)}")


analyze_digital_twin_performance(final_state, simulator.performance_history)

In [ ]:
def run_scenario_analysis(base_simulator: DigitalTwinSimulator) -> Dict[str, Dict]:
    """Run separate what-if simulations and compare their recorded metrics."""
    scenarios = {
        "Equipment failure": simulate_equipment_failure,
        "Storm disruption": simulate_storm_disruption,
        "Peak arrivals": simulate_peak_season,
    }
    results = {}

    print("SCENARIO ANALYSIS — SYNTHETIC WHAT-IF RUNS")
    for scenario_name, scenario_function in scenarios.items():
        scenario_simulator = DigitalTwinSimulator(initialize_digital_twin())
        results[scenario_name] = scenario_function(scenario_simulator)

    print("\nScenario comparison")
    print(f"{'Scenario':<22} {'Completed':>10} {'Mean wait (h)':>15} {'Utilization (%)':>17}")
    print("-" * 68)
    for scenario_name, result in results.items():
        print(
            f"{scenario_name:<22} {result['vessels_served']:>10} "
            f"{result['avg_wait_time']:>15.2f} {result['utilization']:>17.1f}"
        )
    print("Each scenario is a stochastic simulation run; the table is descriptive, not a controlled statistical comparison.")
    return results


def simulate_equipment_failure(simulator: DigitalTwinSimulator) -> Dict[str, float]:
    """Run a scenario with two berths unavailable and fewer tugboats."""
    simulator.state.berths[0].equipment_status["crane"] = False
    simulator.state.berths[2].equipment_status["mooring"] = False
    simulator.resource_system.base_tugboats = 3
    final_scenario_state = simulator.run_simulation(verbose=False)
    metrics = final_scenario_state.performance_metrics
    return {
        "vessels_served": metrics["total_vessels_served"],
        "avg_wait_time": metrics["average_waiting_time"],
        "utilization": metrics["berth_utilization"] * 100,
    }


def simulate_storm_disruption(simulator: DigitalTwinSimulator) -> Dict[str, float]:
    """Run a scenario with a higher modeled storm-probability baseline."""
    simulator.weather_system.storm_probability_base = 0.9
    final_scenario_state = simulator.run_simulation(verbose=False)
    metrics = final_scenario_state.performance_metrics
    return {
        "vessels_served": metrics["total_vessels_served"],
        "avg_wait_time": metrics["average_waiting_time"],
        "utilization": metrics["berth_utilization"] * 100,
    }


def simulate_peak_season(simulator: DigitalTwinSimulator) -> Dict[str, float]:
    """Run a scenario with 50 additional arrivals over the 30-day horizon."""
    additional_vessels = []
    first_new_id = max(vessel.id for vessel in simulator.state.vessels) + 1
    for index in range(50):
        additional_vessels.append(
            Vessel(
                id=first_new_id + index,
                name=f"Peak_Season_Vessel_{index}",
                draft=np.random.uniform(10.5, 15.5),
                length=np.random.uniform(200, 400),
                processing_time=np.random.uniform(3, 8),
                arrival_time=np.random.uniform(0, 720),
                demurrage_cost=np.random.uniform(5000, 15000),
                speed=np.random.uniform(12, 18),
                fuel_consumption=np.random.uniform(40, 60),
            )
        )

    simulator.state.vessels.extend(additional_vessels)
    simulator.state.vessels.sort(key=lambda vessel: vessel.arrival_time)
    simulator.vessel_system = VesselTrackingSubsystem(simulator.state.vessels)
    final_scenario_state = simulator.run_simulation(verbose=False)
    metrics = final_scenario_state.performance_metrics
    return {
        "vessels_served": metrics["total_vessels_served"],
        "avg_wait_time": metrics["average_waiting_time"],
        "utilization": metrics["berth_utilization"] * 100,
    }


scenario_results = run_scenario_analysis(simulator)

In [ ]:
def demonstrate_predictive_analytics(state: DigitalTwinState) -> None:
    """Summarize recent synthetic tide, weather, resource, and workload data."""
    print("PREDICTIVE ANALYTICS DEMONSTRATION")

    if state.tidal_history:
        recent_tidal = state.tidal_history[-100:]
        errors = [
            abs(observation.water_level - observation.predicted_level)
            for observation in recent_tidal
        ]
        confidences = [observation.confidence for observation in recent_tidal]
        print("\nTidal prediction diagnostics (most recent observations):")
        print(f"Mean absolute measured-predicted difference: {np.mean(errors):.3f} m")
        print(f"Maximum measured-predicted difference: {np.max(errors):.3f} m")
        print(f"Mean modeled confidence score: {np.mean(confidences):.3f}")
        print("These diagnostics compare noisy synthetic measurements with noisy predictions.")

    if state.weather_history:
        recent_weather = state.weather_history[-50:]
        high_wind = sum(observation.wind_speed > 25 for observation in recent_weather)
        high_storm_probability = sum(
            observation.storm_probability > 0.7 for observation in recent_weather
        )
        poor_visibility = sum(
            observation.visibility < 1.0 for observation in recent_weather
        )
        print("\nWeather indicators in the most recent observations:")
        print(f"High-wind observations: {high_wind}")
        print(f"Storm-probability observations above 0.7: {high_storm_probability}")
        print(f"Poor-visibility observations: {poor_visibility}")

    if state.resource_history:
        recent_resources = state.resource_history[-48:]
        tugboat_utilization = [
            (6 - observation.tugboats_available) / 6 * 100
            for observation in recent_resources
        ]
        pilot_utilization = [
            (8 - observation.pilots_available) / 8 * 100
            for observation in recent_resources
        ]
        print("\nRecorded resource availability (relative to base counts):")
        print(f"Mean tugboat utilization: {np.mean(tugboat_utilization):.1f}%")
        print(f"Mean pilot utilization: {np.mean(pilot_utilization):.1f}%")
        print("These indicators are monitored; they do not constrain berth allocation.")

    berth_stress = {}
    for berth in state.berths:
        recent_utilization = list(berth.utilization_history)[-24:]
        if recent_utilization:
            berth_stress[berth.id] = float(np.mean(recent_utilization) * 100)
    if berth_stress:
        most_stressed_berth = max(berth_stress, key=berth_stress.get)
        print("\nRecent berth-occupancy indicator:")
        print(
            f"Berth {most_stressed_berth}: "
            f"{berth_stress[most_stressed_berth]:.1f}% mean utilization"
        )

    next_24_hours = [
        vessel
        for vessel in state.vessels
        if state.current_time < vessel.arrival_time <= state.current_time + 24
    ]
    print("\nArrival outlook for the next 24 hours:")
    print(f"Expected arrivals: {len(next_24_hours)}")
    if next_24_hours:
        processing_requirement = sum(
            vessel.processing_time for vessel in next_24_hours
        )
        print(
            "Mean processing time per expected vessel: "
            f"{np.mean([vessel.processing_time for vessel in next_24_hours]):.1f} hours"
        )
        print(
            f"Total processing requirement: {processing_requirement:.1f} berth-hours"
        )
    else:
        print("No arrivals are scheduled in this interval.")


demonstrate_predictive_analytics(final_state)

In [ ]:
def create_digital_twin_dashboard(
    state: DigitalTwinState, performance_history: List[Dict]
) -> None:
    """Display a dashboard of simulated operations and recent conditions."""
    if not performance_history:
        print("No performance observations are available for the dashboard.")
        return

    figure = plt.figure(figsize=(20, 16))
    grid = figure.add_gridspec(4, 4, hspace=0.45, wspace=0.35)
    timestamps = [record["timestamp"] / 24 for record in performance_history]
    completed = [
        record["metrics"]["total_vessels_served"] for record in performance_history
    ]
    waiting_times = [
        record["metrics"]["average_waiting_time"] for record in performance_history
    ]
    utilization = [
        record["metrics"]["berth_utilization"] * 100
        for record in performance_history
    ]

    overview = figure.add_subplot(grid[0:2, 0:2])
    overview.set_title("Completed Vessels and Run Summary")
    overview.plot(timestamps, completed, color="blue", label="Completed vessels")
    overview.fill_between(timestamps, completed, alpha=0.2, color="blue")
    overview.set_xlabel("Simulation time (days)")
    overview.set_ylabel("Completed vessels")
    overview.grid(True, alpha=0.3)
    overview.legend()
    final_metrics = state.performance_metrics
    summary = (
        f"Completed: {int(final_metrics['total_vessels_served'])}\n"
        f"Mean wait: {final_metrics['average_waiting_time']:.2f} h\n"
        f"Final hourly utilization: {final_metrics['berth_utilization'] * 100:.1f}%"
    )
    overview.text(
        0.02,
        0.98,
        summary,
        transform=overview.transAxes,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
    )

    tide_axis = figure.add_subplot(grid[0, 2])
    tide_axis.set_title("Recent Tidal Levels")
    recent_tidal = state.tidal_history[-48:]
    if recent_tidal:
        tide_times = [observation.timestamp for observation in recent_tidal]
        tide_axis.plot(
            tide_times,
            [observation.water_level for observation in recent_tidal],
            label="Measured",
        )
        tide_axis.plot(
            tide_times,
            [observation.predicted_level for observation in recent_tidal],
            linestyle="--",
            label="Predicted",
        )
        tide_axis.legend()
    tide_axis.set_xlabel("Simulation time (hours)")
    tide_axis.set_ylabel("Water level (m)")
    tide_axis.grid(True, alpha=0.3)

    weather_axis = figure.add_subplot(grid[0, 3])
    weather_axis.set_title("Recent Weather")
    recent_weather = state.weather_history[-24:]
    if recent_weather:
        mean_wind = np.mean([item.wind_speed for item in recent_weather])
        maximum_storm_probability = max(
            item.storm_probability for item in recent_weather
        )
        weather_axis.bar(["Mean wind"], [mean_wind], color="orange")
        weather_axis.set_ylabel("Wind speed (knots)")
        weather_axis.text(
            0.5,
            0.85,
            f"Maximum modeled storm probability: {maximum_storm_probability:.2f}",
            transform=weather_axis.transAxes,
            ha="center",
        )
    weather_axis.grid(True, axis="y", alpha=0.3)

    berth_axis = figure.add_subplot(grid[1, 2])
    berth_axis.set_title("Recent Berth Utilization")
    berth_ids = [berth.id for berth in state.berths]
    berth_utilization = []
    for berth in state.berths:
        recent = list(berth.utilization_history)[-24:]
        berth_utilization.append(float(np.mean(recent) * 100) if recent else 0.0)
    berth_bars = berth_axis.bar(berth_ids, berth_utilization, color="steelblue")
    berth_axis.set_xlabel("Berth ID")
    berth_axis.set_ylabel("Utilization (%)")
    berth_axis.set_ylim(0, 100)
    berth_axis.set_xticks(berth_ids)
    for bar, value in zip(berth_bars, berth_utilization):
        berth_axis.text(
            bar.get_x() + bar.get_width() / 2,
            min(value + 1, 98),
            f"{value:.0f}%",
            ha="center",
        )

    resource_axis = figure.add_subplot(grid[1, 3])
    resource_axis.set_title("Latest Resource Availability")
    if state.resource_history:
        latest = state.resource_history[-1]
        resource_names = ["Tugboats", "Pilots", "Stevedores"]
        available = [
            latest.tugboats_available,
            latest.pilots_available,
            latest.stevedores_available,
        ]
        base_counts = [6, 8, 50]
        resource_axis.bar(resource_names, available, color="teal")
        resource_axis.set_ylabel("Available units")
        for index, count in enumerate(available):
            resource_axis.text(index, count, f"{count}/{base_counts[index]}", ha="center")
        resource_axis.tick_params(axis="x", rotation=15)

    waiting_axis = figure.add_subplot(grid[2, :])
    waiting_axis.set_title("Mean Waiting-Time Series")
    waiting_axis.plot(timestamps, waiting_times, color="darkred", label="Hourly mean")
    waiting_axis.fill_between(timestamps, waiting_times, alpha=0.15, color="red")
    waiting_axis.set_xlabel("Simulation time (days)")
    waiting_axis.set_ylabel("Waiting time (hours)")
    waiting_axis.grid(True, alpha=0.3)
    waiting_axis.legend()

    table_axis = figure.add_subplot(grid[3, :])
    table_axis.set_title("Simulation Data Summary")
    table_axis.axis("off")
    completed_vessels = int(final_metrics["total_vessels_served"])
    completion_share = (
        completed_vessels / len(state.vessels) * 100 if state.vessels else 0.0
    )
    table_rows = [
        ["Completed vessels", str(completed_vessels)],
        ["Completed share of modeled arrivals", f"{completion_share:.1f}%"],
        ["Final mean waiting time", f"{final_metrics['average_waiting_time']:.2f} h"],
        ["Final hourly berth utilization", f"{final_metrics['berth_utilization'] * 100:.1f}%"],
        ["Tidal observations", str(len(state.tidal_history))],
        ["Weather observations", str(len(state.weather_history))],
        ["Resource observations", str(len(state.resource_history))],
    ]
    table = table_axis.table(
        cellText=table_rows,
        colLabels=["Metric", "Value"],
        cellLoc="center",
        loc="center",
        colWidths=[0.65, 0.35],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)

    figure.suptitle("Synthetic Port Digital Twin Dashboard", fontsize=18)
    figure.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


create_digital_twin_dashboard(final_state, simulator.performance_history)

## Interpreting the simulation

The plots and printed summaries describe the synthetic scenario generated by this notebook. Completed-vessel counts are cumulative, while berth utilization is the fraction of available berth-hours occupied during a single hourly interval. The reported waiting-time series is the average waiting time among vessels completed by each recorded step; early values can be zero when no vessel has yet completed service.

The scenario experiments alter berth equipment status, the storm-probability baseline, or the number of arrivals. Their outcomes depend on stochastic simulation draws, and the runs are not a paired statistical experiment. The displayed values should therefore be interpreted as outputs of this model and heuristic, not as measured port performance or guaranteed operational recommendations.

### Strengths and limitations

**Strengths**
- The hourly rolling process brings new arrivals and updated synthetic conditions into the allocation cycle.
- Tidal feasibility, weather delay, berth maintenance, equipment status, and schedule conflicts are represented in the assignment logic.
- Separate scenario runs make it possible to inspect how selected modeled disruptions affect the same general allocation procedure.
- The simulation records operational time series that can be visualized and examined after the run.

**Limitations**
- The greedy heuristic is not guaranteed to find a globally optimal schedule, and the sequential priority rule can influence the outcome.
- Stochastic tides and weather are simplified models, not calibrated environmental forecasts.
- Resource histories are monitored but do not restrict assignments; several vessel and berth attributes are likewise outside the allocation constraints.
- The simulation does not connect to live port systems, model data synchronization with physical assets, or establish safety certification.
- Results are specific to the synthetic inputs, assumptions, and random seed in this notebook.

### When this modeling pattern is useful
A rolling simulation can be useful for exploring how a time-dependent allocation heuristic responds to changing arrivals and modeled conditions. A real deployment would require validated operational data, carefully specified constraints, and appropriate operational and safety review. This notebook provides a simulation exercise rather than a deployable port-control system.